# Model Training — Fussball Vorhersagen
Modell A: Nur Spiele mit Aufsteigern
Modell B: Nur Spiele ohne Aufsteiger

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

print('Libraries geladen!')

Libraries geladen!


## 1. Features laden

In [3]:
df = pd.read_csv('data/features.csv', encoding='utf-8')
print(f'Spiele total: {len(df)}')
print(f'Saisons: {df["season"].unique()}')
print(f'Spiele mit Aufsteiger: {((df["is_promoted_home"] == 1) | (df["is_promoted_away"] == 1)).sum()}')
print(f'Spiele ohne Aufsteiger: {((df["is_promoted_home"] == 0) & (df["is_promoted_away"] == 0)).sum()}')

Spiele total: 610
Saisons: [2024 2025]
Spiele mit Aufsteiger: 19
Spiele ohne Aufsteiger: 591


## 2. Features und Zielwert definieren

In [4]:
FEATURES_A = [
    'home_form', 'away_form',
    'home_form_home', 'away_form_away',
    'heimquote', 'home_streak', 'away_streak',
    'home_goal_diff', 'away_goal_diff',
    'is_promoted_home', 'is_promoted_away'
]

FEATURES_B = [
    'home_form', 'away_form',
    'home_form_home', 'away_form_away',
    'heimquote', 'home_streak', 'away_streak',
    'home_goal_diff', 'away_goal_diff'
]

TARGET = 'winner'
label_map = {'HOME_TEAM': 1, 'DRAW': 0, 'AWAY_TEAM': -1}
df['target'] = df[TARGET].map(label_map)

# Datensatz A: NUR Spiele mit Aufsteiger
df_a = df[
    (df['is_promoted_home'] == 1) | (df['is_promoted_away'] == 1)
].dropna(subset=FEATURES_A + [TARGET])

# Datensatz B: NUR Spiele ohne Aufsteiger
df_b = df[
    (df['is_promoted_home'] == 0) & (df['is_promoted_away'] == 0)
].dropna(subset=FEATURES_B + [TARGET])

print(f'Datensatz A (mit Aufsteiger): {len(df_a)} Spiele')
print(f'Datensatz B (ohne Aufsteiger): {len(df_b)} Spiele')

Datensatz A (mit Aufsteiger): 19 Spiele
Datensatz B (ohne Aufsteiger): 591 Spiele


## 3. Train/Test Split

In [5]:
# Split A
train_a = df_a[(df_a['season'] == 2024) |
               ((df_a['season'] == 2025) & (df_a['matchday'] <= 17))]
test_a  = df_a[(df_a['season'] == 2025) & (df_a['matchday'] > 17)]

X_train_a = train_a[FEATURES_A]
y_train_a = train_a['target']
X_test_a  = test_a[FEATURES_A]
y_test_a  = test_a['target']

print(f'Datensatz A — Training: {len(train_a)} | Test: {len(test_a)}')

# Split B
train_b = df_b[(df_b['season'] == 2024) |
               ((df_b['season'] == 2025) & (df_b['matchday'] <= 17))]
test_b  = df_b[(df_b['season'] == 2025) & (df_b['matchday'] > 17)]

X_train_b = train_b[FEATURES_B]
y_train_b = train_b['target']
X_test_b  = test_b[FEATURES_B]
y_test_b  = test_b['target']

print(f'Datensatz B — Training: {len(train_b)} | Test: {len(test_b)}')

Datensatz A — Training: 19 | Test: 0
Datensatz B — Training: 438 | Test: 153


## 4. Baseline

In [ ]:
baseline_a = accuracy_score(y_test_a, [1] * len(y_test_a))
baseline_b = accuracy_score(y_test_b, [1] * len(y_test_b))
print(f'Baseline A (nur Aufsteiger-Spiele): {baseline_a*100:.1f}%')
print(f'Baseline B (ohne Aufsteiger):       {baseline_b*100:.1f}%')

## 5. Modell A — Nur Spiele mit Aufsteigern

In [ ]:
print('=== Modell A: Nur Aufsteiger-Spiele ===\n')

if len(train_a) < 20:
    print('WARNUNG: Zu wenig Trainingsdaten für Modell A!')
    print(f'Nur {len(train_a)} Spiele — Modell wird unzuverlässig sein.')

lr_a = LogisticRegression(max_iter=1000, random_state=42)
lr_a.fit(X_train_a, y_train_a)
lr_a_acc = accuracy_score(y_test_a, lr_a.predict(X_test_a))
print(f'Logistische Regression: {lr_a_acc*100:.1f}%')

rf_a = RandomForestClassifier(n_estimators=100, random_state=42)
rf_a.fit(X_train_a, y_train_a)
rf_a_pred = rf_a.predict(X_test_a)
rf_a_acc  = accuracy_score(y_test_a, rf_a_pred)
print(f'Random Forest:          {rf_a_acc*100:.1f}%')
print()
print(classification_report(y_test_a, rf_a_pred,
      target_names=['Auswärtssieg', 'Unentschieden', 'Heimsieg'],
      zero_division=0))

## 6. Modell B — Nur Spiele ohne Aufsteiger

In [ ]:
print('=== Modell B: Ohne Aufsteiger ===\n')

lr_b = LogisticRegression(max_iter=1000, random_state=42)
lr_b.fit(X_train_b, y_train_b)
lr_b_acc = accuracy_score(y_test_b, lr_b.predict(X_test_b))
print(f'Logistische Regression: {lr_b_acc*100:.1f}%')

rf_b = RandomForestClassifier(n_estimators=100, random_state=42)
rf_b.fit(X_train_b, y_train_b)
rf_b_pred = rf_b.predict(X_test_b)
rf_b_acc  = accuracy_score(y_test_b, rf_b_pred)
print(f'Random Forest:          {rf_b_acc*100:.1f}%')
print()
print(classification_report(y_test_b, rf_b_pred,
      target_names=['Auswärtssieg', 'Unentschieden', 'Heimsieg'],
      zero_division=0))

## 7. Vergleich

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelle = ['Baseline A', 'LR (Aufst.)', 'RF (Aufst.)', 'Baseline B', 'LR (ohne)', 'RF (ohne)']
werte   = [baseline_a, lr_a_acc, rf_a_acc, baseline_b, lr_b_acc, rf_b_acc]
farben  = ['#cccccc', '#2a78d6', '#1baf7a', '#aaaaaa', '#6bb3f5', '#5dd4a8']

axes[0].bar(modelle, [v*100 for v in werte], color=farben)
axes[0].set_title('Genauigkeit der Modelle (%)')
axes[0].set_ylim(0, 100)
axes[0].tick_params(rotation=20)
for i, v in enumerate(werte):
    axes[0].text(i, v*100 + 1, f'{v*100:.1f}%', ha='center', fontsize=8)

# Feature Wichtigkeit Modell B
importances = pd.Series(rf_b.feature_importances_, index=FEATURES_B).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='#1baf7a')
axes[1].set_title('Feature Wichtigkeit Modell B')

plt.tight_layout()
plt.show()

## 8. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, y, title in [
    (axes[0], rf_a_pred, y_test_a, 'Modell A (Aufsteiger-Spiele)'),
    (axes[1], rf_b_pred, y_test_b, 'Modell B (ohne Aufsteiger)')
]:
    cm = confusion_matrix(y, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ausw.', 'Unent.', 'Heim'],
                yticklabels=['Ausw.', 'Unent.', 'Heim'])
    ax.set_title(title)
    ax.set_ylabel('Echtes Ergebnis')
    ax.set_xlabel('Vorhersage')

plt.tight_layout()
plt.show()

## 9. Beide Modelle speichern

In [ ]:
with open('data/model_promoted.pkl', 'wb') as f:
    pickle.dump({'model': rf_a, 'features': FEATURES_A}, f)
print(f'Modell A gespeichert: data/model_promoted.pkl ({rf_a_acc*100:.1f}%)')

with open('data/model_no_promoted.pkl', 'wb') as f:
    pickle.dump({'model': rf_b, 'features': FEATURES_B}, f)
print(f'Modell B gespeichert: data/model_no_promoted.pkl ({rf_b_acc*100:.1f}%)')

## 10. Manuelle Vorhersage testen

In [ ]:
label_map_reverse = {1: 'Heimsieg', 0: 'Unentschieden', -1: 'Auswärtssieg'}

def vorhersage(home_form, away_form, home_form_home, away_form_away,
               heimquote, home_streak, away_streak,
               home_goal_diff, away_goal_diff,
               is_promoted_home=0, is_promoted_away=0):

    if is_promoted_home == 1 or is_promoted_away == 1:
        model_data  = pickle.load(open('data/model_promoted.pkl', 'rb'))
        modell_name = 'Modell A (Aufsteiger-Spiele)'
    else:
        model_data  = pickle.load(open('data/model_no_promoted.pkl', 'rb'))
        modell_name = 'Modell B (ohne Aufsteiger)'

    model    = model_data['model']
    features = model_data['features']

    beispiel = pd.DataFrame([{
        'home_form':        home_form,
        'away_form':        away_form,
        'home_form_home':   home_form_home,
        'away_form_away':   away_form_away,
        'heimquote':        heimquote,
        'home_streak':      home_streak,
        'away_streak':      away_streak,
        'home_goal_diff':   home_goal_diff,
        'away_goal_diff':   away_goal_diff,
        'is_promoted_home': is_promoted_home,
        'is_promoted_away': is_promoted_away
    }])[features]

    ergebnis = model.predict(beispiel)[0]
    probs    = model.predict_proba(beispiel)[0]

    print(f'Modell: {modell_name}')
    print(f'Vorhersage: {label_map_reverse[ergebnis]}')
    print('Wahrscheinlichkeiten:')
    for klasse, prob in zip(model.classes_, probs):
        print(f'  {label_map_reverse[klasse]}: {prob*100:.1f}%')


# Beispiel 1: Bayern vs Dortmund (keine Aufsteiger)
print('Bayern (Heim) vs Dortmund (Auswärts):')
vorhersage(
    home_form=12, away_form=6,
    home_form_home=10, away_form_away=4,
    heimquote=0.7, home_streak=3, away_streak=0,
    home_goal_diff=1.5, away_goal_diff=-0.5
)

print()

# Beispiel 2: Bayern vs Hamburger SV (Aufsteiger)
print('Bayern (Heim) vs Hamburger SV (Aufsteiger):')
vorhersage(
    home_form=12, away_form=3,
    home_form_home=10, away_form_away=2,
    heimquote=0.7, home_streak=3, away_streak=0,
    home_goal_diff=1.5, away_goal_diff=-0.8,
    is_promoted_away=1
)